### Tidy Data Project ### 

In this project, I take the mutant moneyball dataset and apply tidy data principles to discern trends among the valuation of the appearance of X-Men characters. This first block of code imports pandas and displays the csv file in a dataframe. The dataframe is then melted, including shifting the TotalValue columns into a single column, each with an individual observation in the form of the valuation. 

In [ ]:
import pandas as pd 

df = pd.read_csv("mutant_moneyball.csv") # loads dataset from csv file
df_tidy = df.melt(id_vars=['Member'], 
                  value_vars=['TotalValue60s_heritage', 'TotalValue70s_heritage', 'TotalValue80s_heritage', 
                              'TotalValue90s_heritage', 'TotalValue60s_ebay', 'TotalValue70s_ebay','TotalValue80s_ebay',
                              'TotalValue90s_ebay',	'TotalValue60s_wiz', 'TotalValue70s_wiz', 'TotalValue80s_wiz',
                              'TotalValue90s_wiz', 'TotalValue60s_oStreet', 'TotalValue70s_oStreet', 'TotalValue80s_oStreet',
                              'TotalValue90s_oStreet'], # list of all columns to be melted and tidied (contains decade and source of evaluation in the same column)
                 var_name= 'Evaluator',
                 value_name='Valuation') # melts table to align with tidy data principles 
df_tidy

The TotalValue values are not complianet with tidy data principles, since they contain both the evaluator and a decade. This is fixed by using the string plit function. 'TotalValue' is removed and the valuations are converted to strings, and the dollar signs are dropped to prevent null values. 

In [4]:
df_tidy[['Decade', 'Source']] = df_tidy['Evaluator'].str.split("_", expand=True) # splits Evaluator column into Source (of evaluation) and Decade
df_tidy['Decade'] = df_tidy['Decade'].str.replace("TotalValue", '', regex=False) # removes TotalValue from decade and evaluator columns
df_tidy['Valuation'] = (
    df_tidy['Valuation']
    .astype(str)
    .replace('[\$,\s]', '', regex=True)) # removes dollar sign to allow for proper visualization without causing valuations to be read as strings
df_tidy['Decade'] = (
    df_tidy['Decade']
    .replace('60s', '1960s', regex=True)
    .replace('70s', '1970s', regex=True)
    .replace('80s', '1980s', regex=True)
    .replace('90s', '1990s', regex=True)) # adds '19' to the decade values for context and readability
df_tidy['Valuation'] = pd.to_numeric(df_tidy['Valuation'], errors='coerce') # adjusts values to prevent null values when making evaluations uniform

<>:6: SyntaxWarning: invalid escape sequence '\$'
<>:6: SyntaxWarning: invalid escape sequence '\$'
/var/folders/9f/f2r2bq793nqd3zrv8mxk8plm0000gn/T/ipykernel_61213/20836867.py:6: SyntaxWarning: invalid escape sequence '\$'
  .replace('[\$,\s]', '', regex=True)) # removes dollar sign to allow for proper visualization without causing valuations to be read as strings


While the names of both the X-Men and Evaluators are optimal for the dataset, they are not optimal for readability and being displayed in a streamlit app. Renaming them allows for improved visualization. 

In [5]:
new_sources = {
    "heritage" : "Heritage",
    "ebay" : "Ebay",
    "wiz" : "Wiz",
    "ostreet" : "O Street"} # renames sources of evaluation to more readable form 

members_renamed = {
    'warrenWorthington' : 'Warren Worthington',
    'hankMcCoy' : 'Hank McCoy',
    'scottSummers' : 'Scott Summers',
    'bobbyDrake' : 'Bobby Drake',
    'jeanGrey' : 'Jean Grey',
    'alexSummers' : 'Alex Summers',
    'lornaDane' : 'Lorna Dane',
    'ororoMunroe' : 'Ororo Munroe',
    'kurtWagner' : 'Kurt Wagner',
    'loganHowlett' : 'Logan Howlett',
    'peterRasputin' : 'Peter Rasputin',
    'seanCassidy' : 'Sean Cassidy',
    'shiroYoshida' : 'Shiro Yoshida',
    'johnProudstar' : 'John Proudstar',
    'kittyPryde' : 'Kitty Pryde',
    'annaMarieLeBeau' : 'Anna Marie LeBeau',
    'rachelSummers' : 'Rachel Summers',
    'ericMagnus' : 'Eric Magnus',
    'alisonBlaire' : 'Alison Blaire',
    'longshot' : 'Longshot',
    'jonathanSilvercloud': 'Jonathon Silvercloud',
    'remyLeBeau' : 'Remy LeBeau',
    'jubilationLee' : 'Jubiliation Lee',
    'lucasBishop' : 'Lucas Bishop',
    'betsyBraddock' : 'Betsy Braddock',
    'charlesXavier' : 'Charles Xavier'} # change names from canalCase for readability and to prevent null values

This final step of EDA cleans up the melted dataset, including dropping the Evaluator column (now redundant due to the string split columns). Finally, the above dictionaries are mapped onto the 'Source' and 'Member' columns, creating the final dataframe. 

In [ ]:
final_df_tidy = df_tidy.drop(columns=['Evaluator']) # drops now-redundant evaluator column
final_df_tidy["Source"] = final_df_tidy["Source"].str.lower().replace(new_sources) # replaces new sources to clean the dataset 
final_df_tidy["Member"] = final_df_tidy["Member"].replace(members_renamed) # adds the renamed members to the cleaned dataset
print(final_df_tidy) # displays cleaned dataset

### DATA CLEANING ### 
In this first step, I took actions to make the dataset as readable and clean as possible. Since both platform and decade were captured in the same column, I employed tidy data principles to split up these values, making both individuals observations in the cleaned data set. I then dropped the original, now redundant column. I also renamed both the sources of data and the X-Men within the data itself to increase readability through dictionaries that reassigned values. In addition, I expanded my skillset and added code that made sure that no dollar signs or commas were included to increase readability. Finally, I printed the new clean dataset that followed tidy data principles with each cell containing a single value, each member of the X-Men forming their own row, and each variable forming a single column. 

### 1st Four Charts ### 
These first four charts are critical because 3/4 platforms track accumulated sales for character appearances in comics from a certain decade. Over time, sales go down, showing the value of older comic book appearances. Using an average of all X-Men in a decade for each evaluator, one can see the average total value sold on each platform across all characters. Together, all four graphs provide a complete picture as to the total average sales of character appearances across X-Men on each platform across four decades. 

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
# These 4 graphs are grouped together to show change over time as denoted by the Y axis 
fdt = final_df_tidy 
plt.title("Average Accumulated Value of Sales from 3 Evaluators for 1960s Comic Book Appearances") # Uses the three evaluators with metrics in similar magnitudes; all based on average accumulated valueof sales from certain decades 
sixties_df = fdt[(fdt['Decade']=='1960s') & ~(fdt['Source']=='Heritage')] # Heritage uses highest sale, not total accumulated value
sns.boxplot(
    data=sixties_df,
    x='Source',
    y='Valuation') # uses boxplot to show averages, quartiles, standard deviations, and outliers; avoids creation of false negative values of violin plots 
plt.xlabel("Platform") # axes renamed 
plt.ylabel('Total Valuation of Sales ($)')
plt.show()

plt.title("Average Accumulated Value of Sales from 3 Evaluators for 1970s Comic Book Appearances") # same as above graph 
seventies_df = fdt[(fdt['Decade']=='1970s') & ~(fdt['Source']=='Heritage')]
sns.boxplot(
    data=seventies_df,
    x='Source',
    y='Valuation')
plt.xlabel("Platform")
plt.ylabel("Total Average Valuation of Sales ($)")
plt.show()

plt.title("Average Accumulated Value of Sales from 3 Evaluators for 1980s Comic Book Appearances")
eighties_df = fdt[(fdt['Decade']=='1980s') & ~(fdt['Source']=='Heritage')]
sns.boxplot(
    data=eighties_df,
    x='Source',
    y='Valuation')
plt.xlabel("Platform")
plt.ylabel("Total Average Valuation of Sales ($)")
plt.show()

fdt = final_df_tidy
plt.title("Average Accumulated Value of Sales from 3 Evaluators for 1990s Comic Book Appearances")
nineties_df = fdt[(fdt['Decade']=='1990s') & ~(fdt['Source']=='Heritage')] 
sns.boxplot(
    data=nineties_df,
    x='Source',
    y='Valuation')
plt.xlabel("Platform")
plt.ylabel("Total Average Valuation of Sales ($)")
plt.show()

### Heritage Chart ### 

Heritage's values are much higher. This is because Heritage tracks the highest individual sale of an appearnce from a certain decade for each member of the X-Men. Based on this difference, I created a separate scatterplot color-coded scatterplot. As more characters are added, more data points appear. Values decline over time, as a 1960s appearance of Professor X is much more valuable than a 1990s appearance of Kitty Pryde. This graph optimizes readability while providing a stronger understanding of scale. 

In [ ]:
heritage_df = fdt[fdt['Source'] == 'Heritage'].copy() # allows for modifications without affecting original data

plt.figure(figsize=(12, 6)) # changes sidze to accomodate many data points 
sns.scatterplot(
    data=heritage_df,
    x='Member',       
    y='Valuation',
    hue='Decade') # color-coded scatterplot used show change in individual value over time; Heritage values are individual highs for each decade
# More X-Men are present over time because most characters were added later 
plt.title('Heritage Value of X-Men Members Across Decades')
plt.xlabel("X-Men Members")
plt.ylabel("Heritage Valuation ($)")
plt.xticks(rotation=45, ha='right') # drastically increases readability thorugh adjusted orientation of x-axis labels 
plt.grid(axis='y', linestyle='--', alpha=0.6) # increases readability of graph through better spacing  
plt.tight_layout()
plt.show()

### Last 3 Graphs ###
In an inverse of the previous chart, the individuals platforms are made a subset of the dataframe and the decades are categorized on the x-axis, allowing for an understanding of average valuation across time. This shows the decline of average accumulated value over time from each source. This allows for isolation to see differences in each of the three accumulation platforms used in the dataset. 

In [ ]:
ebay_df = final_df_tidy[final_df_tidy['Source']=='Ebay'] # Shows Evaluation of prices over time 
sns.barplot(
    data=ebay_df,
    x='Decade',
    y='Valuation')
plt.title('Valuation ($) of X-Men Members Across Decades')
plt.xlabel('Decade')
plt.ylabel('Average Valuation Price of X-Men Members')

In [ ]:
ebay_df = final_df_tidy[final_df_tidy['Source']=='Wiz'] # Shows Evaluation of prices over time 
sns.barplot(
    data=ebay_df,
    x='Decade',
    y='Valuation')
plt.title('Valuation ($) of X-Men Members Across Decades')
plt.xlabel('Decade')
plt.ylabel('Average Valuation Price of X-Men Members')

In [ ]:
o_street_df=final_df_tidy[final_df_tidy['Source']=='O Street']
sns.barplot(
    data=o_street_df,
    x='Decade',
    y='Valuation')
plt.title('Valuation ($) of X-Men Members Across Decades')
plt.xlabel('Decade')
plt.ylabel('Average Valuation Price of X-Men Members')

### PIVOT TABLES AND CONCLUSION ### 
To differentiate my pivot tables, I tracked the median instead of the mean. This provides a contrast to see how valuable the 50th percentile is when not affected by the older, more popular members of the X-Men (Professor X, Cyclops, etc.). Overall, the pivot tables provide a companion to the graphs that compare valuation across both platform of sale and decade of appearance. Together with the clean dataset, they provide a picture of the meaning of the data and the value of comic book appearances. 

In [ ]:
final_df_tidy['Valuation'] = final_df_tidy['Valuation'].replace('[\$,]', '', regex=True)
final_df_tidy['Valuation'] = pd.to_numeric(final_df_tidy['Valuation'], errors='coerce')

summary_table1 = final_df_tidy.pivot_table(
    index = 'Source', 
    columns = 'Decade', 
    values = 'Valuation', 
    aggfunc = 'median')
print(summary_table1)

summary_table1 = final_df_tidy.pivot_table(
    index = 'Decade', 
    columns = 'Source', 
    values = 'Valuation', 
    aggfunc = 'median')
print(summary_table1)